# 04 — Silver: For-Hire Vehicle (FHV)

Reads FHV records from Bronze (`tlc_bronze.fhv_raw`), applies data quality rules
via **flag-based annotation** (no silent drops), enriches with zone metadata,
builds the normalized Silver schema, and writes:
- **Valid records** → `tlc_silver.trips_fhv` (with `quality_flags` preserved)
- **Invalid records** → `tlc_audit.quarantine` (with `_rejection_reason`)

In [1]:
import sys
sys.path.insert(0, '/home/jovyan/work')

from src.spark_utils import get_spark, read_mongo, write_mongo
from src.transformations.tlc_rules import (
    FHV_RULES, apply_quality_flags, route_quarantine, print_rejection_summary
)
from src.transformations.enrichment import load_zone_lookup, enrich_trip_locations
from src.transformations.schema import build_fhv_silver
from core.config.settings import settings
from core.audit.control_manager import ControlManager, ExecutionStatus
from core.storage.mongo_client import get_audit_db
import pyspark.sql.functions as F
import datetime

YEARS_TO_PROCESS = [2023, 2024, 2025]
print("Imports OK.")

Imports OK.


In [2]:
spark = get_spark(app_name="TLC_Silver_FHV")

control = ControlManager("silver_fhv")
execution = control.start({"years": YEARS_TO_PROCESS, "vehicle_type": "fhv"})
run_id = execution.execution_id
print(f"Execution ID (Silver run_id): {run_id}")

2026-07-19 03:10:55 | INFO     | tlc.spark_utils | [SPARK] Session ready | version=3.4.1 master=local[*]
2026-07-19 03:10:55 | INFO     | tlc.audit.control_manager | [AUDIT] Execution started | pipeline=silver_fhv id=389c4e49
Execution ID (Silver run_id): 389c4e49


## Idempotency Purge

Ensures this notebook can be re-run safely without duplicating data by clearing previous artifacts for this pipeline.

In [ ]:
from core.storage.mongo_client import get_audit_db, get_silver_db
from core.config.settings import settings

client = get_silver_db()
# 1. Purge Silver collection for fhv
get_silver_db()["trips_fhv"].delete_many({})

# 2. Purge Quarantine records generated by this pipeline
get_audit_db()["quarantine"].delete_many({"pipeline": "silver_fhv"})

print("Idempotency purge complete for fhv. Safe to run.")


In [3]:
df_bronze = read_mongo(spark, settings.MONGO_DB_BRONZE, "fhv_raw")

if "_meta" in df_bronze.columns:
    df_bronze = df_bronze.filter(F.year("pickup_datetime").isin(YEARS_TO_PROCESS))

# records_in will be evaluated after caching df_flagged


2026-07-19 03:11:00 | INFO     | tlc.spark_utils | [SPARK] Read from MongoDB tlc_bronze.fhv_raw


In [4]:
df_flagged = apply_quality_flags(df_bronze, FHV_RULES)
records_in = df_flagged.count()
print(f"Records read from Bronze: {records_in:,}")

valid_df, rejected_df = route_quarantine(df_flagged)

records_valid    = valid_df.count()
records_rejected = rejected_df.count()

print(f"Valid records   : {records_valid:,}")
print(f"Rejected records: {records_rejected:,}")
print(f"Quarantine rate : {records_rejected / records_in * 100:.2f}%" if records_in else "N/A")

print_rejection_summary(rejected_df)

control.log_quality_check(
    check_name="fhv_quality_flags",
    dataset=f"fhv_raw_years_{YEARS_TO_PROCESS}",
    records_checked=records_in,
    records_failed=records_rejected,
)

2026-07-19 03:11:00 | INFO     | tlc.transformations.tlc_rules | [RULES] Quality flags applied | rules=['valid_pickup_zone', 'valid_dropoff_zone', 'valid_time_order', 'valid_dispatching_base']
Records read from Bronze: 58,536,509
Valid records   : 58,535,374
Rejected records: 1,135
Quarantine rate : 0.00%
2026-07-19 03:12:10 | INFO     | tlc.transformations.tlc_rules | [RULES] Total records rejected: 1,135
2026-07-19 03:12:30 | INFO     | tlc.transformations.tlc_rules |          ↳ 'Dispatching base num must start with 'B'.': 798
2026-07-19 03:12:30 | INFO     | tlc.transformations.tlc_rules |          ↳ 'Dropoff datetime must be after pickup datetime.': 337
2026-07-19 03:12:30 | INFO     | tlc.audit.control_manager | [QUALITY] fhv_quality_flags | dataset=fhv_raw_years_[2023, 2024, 2025] status=WARNING failure_rate=0.00%


QualityCheckResult(check_id='389c4e49_fhv_quality_flags', check_name='fhv_quality_flags', dataset='fhv_raw_years_[2023, 2024, 2025]', status=<QualityStatus.WARNING: 'WARNING'>, records_checked=58536509, records_passed=58535374, records_failed=1135, failure_rate=1.938960862869359e-05, details={})

In [5]:
if records_rejected > 0:
    # OPTIMIZATION: Distributed PySpark write instead of driver collect()
        seen_cols = set()
    raw_cols = []
    for c in rejected_df.columns:
        if c not in ("_rejection_reason", "quality_flags", "_meta") and c.lower() not in seen_cols:
            raw_cols.append(c)
            seen_cols.add(c.lower())
    
    quarantine_df_mapped = rejected_df.select(
        F.current_timestamp().alias("quarantined_at"),
        F.lit(run_id).alias("silver_run_id"),
        F.col("_meta.run_id").alias("bronze_run_id"),
        F.col("_meta.source_file").alias("source_file"),
        F.lit("silver_fhv").alias("pipeline"),
        F.col("_rejection_reason").alias("reason"),
        F.col("quality_flags").alias("quality_flags"),
        F.struct(*[F.col(c) for c in raw_cols]).alias("raw_record")
    )
    
    write_mongo(quarantine_df_mapped, settings.MONGO_DB_AUDIT, "quarantine", mode="append")
    print(f"Quarantined {records_rejected:,} records into tlc_audit.quarantine (Distributed)")
else:
    print("No records quarantined.")


2026-07-19 03:13:31 | INFO     | tlc.spark_utils | [SPARK] Wrote Unknown rows → MongoDB tlc_audit.quarantine (mode=append)
Quarantined 1,135 records into tlc_audit.quarantine (Distributed)


In [6]:
zone_df = load_zone_lookup(spark)
valid_df = enrich_trip_locations(valid_df, zone_df)
print("Zone enrichment complete.")

2026-07-19 03:13:32 | INFO     | tlc.transformations.enrichment | [ENRICH] Zone lookup loaded: 265 zones from /home/jovyan/work/data/raw/lookup/taxi_zone_lookup.csv
2026-07-19 03:13:32 | INFO     | tlc.transformations.enrichment | [ENRICH] Location IDs enriched with Borough and Zone names.
Zone enrichment complete.


In [7]:
silver_df = build_fhv_silver(valid_df, run_id=run_id)
n_written = write_mongo(silver_df, settings.MONGO_DB_SILVER, "trips_fhv", mode="append", num_rows=records_valid)
print(f"Rows written to tlc_silver.trips_fhv: {n_written:,}")

2026-07-19 03:13:32 | INFO     | tlc.transformations.schema | [SCHEMA] FHV Silver schema applied (with bronze lineage + quality_flags).
2026-07-19 03:21:42 | INFO     | tlc.spark_utils | [SPARK] Wrote 58,535,374 rows → MongoDB tlc_silver.trips_fhv (mode=append)
Rows written to tlc_silver.trips_fhv: 58,535,374


In [8]:
print(f"╔══════════════════════════════════════════╗")
print(f"║  Volumetric Traceability Report          ║")
print(f"╠══════════════════════════════════════════╣")
print(f"║  Bronze records in  : {records_in:>16,}  ║")
print(f"║  Silver records out : {n_written:>16,}  ║")
print(f"║  Quarantine records : {records_rejected:>16,}  ║")
print(f"║  Delta (must be 0)  : {records_in - n_written - records_rejected:>16,}  ║")
print(f"╚══════════════════════════════════════════╝")
assert records_in == n_written + records_rejected, "DATA LOSS DETECTED — investigate immediately!"

control.end(
    ExecutionStatus.SUCCESS,
    {
        "records_read_from_bronze": records_in,
        "records_written_to_silver": n_written,
        "records_quarantined": records_rejected,
        "quarantine_rate_pct": round(records_rejected / records_in * 100, 4) if records_in else 0,
    },
)

╔══════════════════════════════════════════╗
║  Volumetric Traceability Report          ║
╠══════════════════════════════════════════╣
║  Bronze records in  :       58,536,509  ║
║  Silver records out :       58,535,374  ║
║  Quarantine records :            1,135  ║
║  Delta (must be 0)  :                0  ║
╚══════════════════════════════════════════╝
2026-07-19 03:21:43 | INFO     | tlc.audit.control_manager | [AUDIT] Execution finished | id=389c4e49 status=SUCCESS duration=647.3s
2026-07-19 03:21:43 | INFO     | tlc.audit.control_manager | [AUDIT] Report saved → /home/jovyan/work/data/audit/executions/20260719_032143_389c4e49_silver_fhv.json
2026-07-19 03:21:43 | INFO     | tlc.audit.control_manager | [AUDIT] Report inserted into MongoDB tlc_audit.pipeline_runs (id=389c4e49)


ExecutionRecord(pipeline_name='silver_fhv', input_parameters={'years': [2023, 2024, 2025], 'vehicle_type': 'fhv'}, execution_id='389c4e49', start_time=datetime.datetime(2026, 7, 19, 3, 10, 55, 799652), end_time=datetime.datetime(2026, 7, 19, 3, 21, 43, 53987), status=<ExecutionStatus.SUCCESS: 'SUCCESS'>, output_summary={'records_read_from_bronze': 58536509, 'records_written_to_silver': 58535374, 'records_quarantined': 1135, 'quarantine_rate_pct': 0.0019}, quality_checks=[QualityCheckResult(check_id='389c4e49_fhv_quality_flags', check_name='fhv_quality_flags', dataset='fhv_raw_years_[2023, 2024, 2025]', status=<QualityStatus.WARNING: 'WARNING'>, records_checked=58536509, records_passed=58535374, records_failed=1135, failure_rate=1.938960862869359e-05, details={})], errors=[])

In [9]:
import json
print(json.dumps(control.get_report(), indent=2, default=str))

{
  "execution_id": "389c4e49",
  "pipeline_name": "silver_fhv",
  "status": "SUCCESS",
  "start_time": "2026-07-19T03:10:55.799652",
  "end_time": "2026-07-19T03:21:43.053987",
  "duration_seconds": 647.25,
  "input_parameters": {
    "years": [
      2023,
      2024,
      2025
    ],
    "vehicle_type": "fhv"
  },
  "output_summary": {
    "records_read_from_bronze": 58536509,
    "records_written_to_silver": 58535374,
    "records_quarantined": 1135,
    "quarantine_rate_pct": 0.0019
  },
  "quality_checks": [
    {
      "check_id": "389c4e49_fhv_quality_flags",
      "check_name": "fhv_quality_flags",
      "dataset": "fhv_raw_years_[2023, 2024, 2025]",
      "status": "WARNING",
      "records_checked": 58536509,
      "records_passed": 58535374,
      "records_failed": 1135,
      "failure_rate": 1.9e-05,
      "details": {}
    }
  ],
  "quality_passed": 0,
  "errors": []
}
